# 視覚的アテンション（Visual Attention）を用いた画像キャプショニング

**学習目標**
- データセットの読み込みと前処理方法を学ぶ
- キャプションをトークン化する方法を学ぶ
- アテンション機構を備えた Encoder-Decoder モデルを構築する
- トレーニング済みモデルを使用してキャプションを予測・生成する

このノートブックでは、[Show, Attend and Tell: Neural Image Caption Generation with Visual Attention](https://arxiv.org/abs/1502.03044) に類似したモデルアーキテクチャを使用し、アテンションベースの画像キャプショニング モデルを構築します。

このノートブックはエンドツーエンドの例です。トレーニング データセットには、COCO の大規模な物体検出、セグメンテーション、キャプショニング データセットを使用します。

## 準備

In [ ]:
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import time
from textwrap import wrap

import keras
import matplotlib.pylab as plt
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
from keras import Input
from keras.layers import (
    GRU,
    Add,
    Attention,
    Concatenate,
    Dense,
    Embedding,
    LayerNormalization,
    Rescaling,
    Reshape,
    Resizing,
    StringLookup,
    TextVectorization,
)

print(keras.version())

## データセットの読み込みと準備

TensorFlow Datasets の機能を使用して、[COCO captions](https://www.tensorflow.org/datasets/catalog/coco_captions) データセットを読み込みます。
このバージョンには、COCO 2014 の画像、バウンディング ボックス、ラベル、キャプションが含まれており、Karpathy and Li (2015) によって定義されたサブセットに分割されています。また、元のデータセットの一部のデータ品質の問題（元のデータセットの一部の画像にキャプションがなかったなど）に対処しています。

まず、いくつかの定数を定義しましょう。<br>
このラボでは、特徴量抽出器として `keras.applications` の事前トレーニング済み [InceptionResNetV2](https://keras.io/api/applications/inceptionresnetv2/) モデルを使用するため、一部の定数は InceptionResNetV2 モデルの定義に基づいています。<br>
そのため、別のベースモデルを使用したい場合は、これらの定数も同様に変更してください。

`keras.applications` は、一般的で安定した画像モデルを提供する事前トレーニング済みモデル リポジトリです。<br>
一部のプラットフォームではコンパイル済みのモデルグラフとパラメータのみを含むモデルが提供されるのに対し、`keras.applications` はモデルのメタデータを含んでおり、モデルの動作にアクセスして制御できるため柔軟性があります。<br>
したがって、レイヤーのメタデータにアクセスすることで、モデルの最終レイヤーの出力（CNN モデルのフラット化された 1D Tensor 出力など）だけでなく、中間レイヤーの出力（中間の 3D Tensor など）も取得できます。

In [ ]:
# 精度/速度を調整するためにこれらを変更します
VOCAB_SIZE = 20000  # 収束を早めるため単語数を減らします
ATTENTION_DIM = 512  # Attention の Dense レイヤーのサイズ
WORD_EMBEDDING_DIM = 128

# InceptionResNetV2 は (299, 299, 3) の画像を入力として受け取り、
# (8, 8, 1536) の形状の特徴量を返します
FEATURE_EXTRACTOR = keras.applications.inception_resnet_v2.InceptionResNetV2(
    include_top=False, weights="imagenet"
)
IMG_HEIGHT = 299
IMG_WIDTH = 299
IMG_CHANNELS = 3
FEATURES_SHAPE = (8, 8, 1536)

resize = Resizing(IMG_HEIGHT, IMG_WIDTH)
rescale = Rescaling(1.0 / 255)

### フィルタリングと前処理
ここではデータセットの前処理を行います。以下の関数は次の処理を行います。
- 画像を (`IMG_HEIGHT`, `IMG_WIDTH`) の形状にリサイズ
- ピクセル値を [0, 255] から [0, 1] にリスケール
- 画像（`image_tensor`）とキャプション（`caption`）の辞書を返す

**注**: このデータセットはローカル環境に保存するには大きすぎます。そのため、us-central1 にある公開 GCS バケットに保存されています。
米国以外のノートブックからアクセスする場合、(a) 処理が遅くなり、(b) ネットワーク料金が発生する可能性があります。

In [ ]:
GCS_DIR = "gs://asl-public-data/data/tensorflow_datasets/"
BUFFER_SIZE = 1000


def get_image_label(example):
    caption = example["captions"]["text"][0]  # 画像あたり最初のキャプションのみ使用
    img = example["image"]
    img = resize(img)
    img = rescale(img)
    return {"image_tensor": img, "caption": caption}


trainds = tfds.load("coco_captions", split="train", data_dir=GCS_DIR)

trainds = trainds.map(
    get_image_label, num_parallel_calls=tf.data.AUTOTUNE
).shuffle(BUFFER_SIZE)
trainds = trainds.prefetch(buffer_size=tf.data.AUTOTUNE)

### 可視化
データセット内の画像とサンプルのキャプションを確認してみましょう。

In [ ]:
f, ax = plt.subplots(1, 4, figsize=(20, 5))
for idx, data in enumerate(trainds.take(4)):
    ax[idx].imshow(data["image_tensor"].numpy())
    caption = "\n".join(wrap(data["caption"].numpy().decode("utf-8"), 30))
    ax[idx].set_title(caption)
    ax[idx].axis("off")

## テキストの前処理

文の開始（`<start>`）と終了（`<end>`）を表す特殊トークンを追加します。<br>
ここで開始トークンと終了トークンを追加するのは、Encoder-Decoder モデルを使用しており、予測時にキャプション生成を開始するために `<start>` を使用し、キャプションの長さが可変であるため、`<end>` トークンが現れた時点で予測を終了するためです。

その後、さらなる前処理のためにキャプションの完全なリストを作成します。

In [ ]:
def add_start_end_token(data):
    start = keras.ops.convert_to_tensor("<start>")
    end = keras.ops.convert_to_tensor("<end>")
    data["caption"] = tf.strings.join(
        [start, data["caption"], end], separator=" "
    )
    return data


trainds = trainds.map(add_start_end_token)

## キャプションの前処理とトークン化

[TextVectorization](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) レイヤーを使用して、テキスト キャプションを以下の手順で整数のシーケンスに変換します。

* [adapt](https://github.com/keras-team/keras/blob/adbfd13426a0da9864d9a0fcd5be5eed74ca341f/keras/src/layers/preprocessing/text_vectorization.py#L387) を使用してすべてのキャプションを反復処理し、キャプションを単語に分割して、上位 `VOCAB_SIZE` 個の単語からなる語彙を計算します。
* 各単語を語彙内のインデックスにマッピングすることで、すべてのキャプションをトークン化します。すべての出力シーケンスは `MAX_CAPTION_LEN` の長さにパディングされます。ここではこのデータセットに十分な `64` という数値を直接指定していますが、データセット内の非常に長い文を切り捨てたくない場合は、データセット全体を処理してこの値を算出する必要がある点に注意してください。

**注**: この処理には約 10 分かかります。

**演習**: [ドキュメント](https://keras.io/api/layers/preprocessing_layers/text/text_vectorization/) を参考に、`TextVectorization` レイヤーを完成させてください。

**ヒント**: `VOCAB_SIZE` 変数、`MAX_CAPTION_LEN` 変数、および `standardize` 関数を使用できます。

In [ ]:
MAX_CAPTION_LEN = 64


# TextVectorization のデフォルトの標準化をオーバーライドして
# 「<>」文字を保持し、<start> と <end> のトークンを維持します。
def standardize(inputs):
    inputs = tf.strings.lower(inputs)
    return tf.strings.regex_replace(
        inputs, r"[!\"#$%&\(\)\*\+.,-/:;=?@\[\\\]^_`{|}~]?", ""
    )


# 語彙から最も頻出する単語を選択し、句読点などを削除します。
# TODO: TextVectorization レイヤーを完成させてください。
tokenizer = TextVectorization(...)

tokenizer.adapt(trainds.map(lambda x: x["caption"]))


サンプルのテキストをトークン化してみましょう。

In [ ]:
tokenizer(["<start> This is a sentence <end>"])

In [ ]:
sample_captions = []
for d in trainds.take(5):
    sample_captions.append(d["caption"].numpy())

In [ ]:
sample_captions

In [ ]:
print(tokenizer(sample_captions))

すべての文が同じトークン（例:「3」と「4」）で始まり、終わっていることに注目してください。これらの値は、それぞれ開始トークンと終了トークンを表しています。

ID を元のテキストに戻すこともできます。

In [ ]:
for wordid in tokenizer([sample_captions[0]])[0]:
    print(tokenizer.get_vocabulary()[wordid], end=" ")

また、`StringLookup` レイヤーを使用して、単語 <-> インデックスの変換器を作成できます。

In [ ]:
# ルックアップ テーブル: 単語 -> インデックス
word_to_index = StringLookup(
    mask_token="", vocabulary=tokenizer.get_vocabulary()
)

# ルックアップ テーブル: インデックス -> 単語
index_to_word = StringLookup(
    mask_token="", vocabulary=tokenizer.get_vocabulary(), invert=True
)

### トレーニング用の tf.data データセットの作成
適合させたトークン化をすべてのサンプルに適用し、トレーニング用の tf.data Dataset を作成しましょう。

ここでは、特徴量キャプションからテキストをシフトさせてラベルを作成していることにも注意してください。<br>
入力キャプションが `"<start> I love cats <end>"` の場合、そのラベルは `"I love cats <end> <padding>"` になります。<br>
これにより、モデルは `<start>` から `I` を予測することを学習できます。

データセットはタプルを返す必要があり、第 1 要素は特徴量（`image_tensor` と `caption`）、第 2 要素はラベル（ターゲット）です。

**演習**: データセットを作成する `create_ds_fn` を定義してください。以下のステップを含める必要があります。
- `keras.ops.roll` を使用してキャプションを 1 ステップロール（シフト）します。ロールされたキャプションがターゲットとして使用されます。
- ロールされたキャプションにゼロ値（パディング）を追加し、元のキャプションと同じ長さのベクトルを作成します。
- 特徴量（画像と元のキャプション）およびターゲット（ロールされたキャプション）を返します。

In [ ]:
BATCH_SIZE = 32


def create_ds_fn(data):
    img_tensor = data["image_tensor"]
    caption = tokenizer(data["caption"])

    # TODO: ステップを定義してください。
    ...
    return ...


batched_ds = (
    trainds.map(create_ds_fn)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(buffer_size=tf.data.AUTOTUNE)
)

いくつかのサンプルを確認してみましょう。

In [ ]:
for (img, caption), label in batched_ds.take(2):
    print(f"Image shape: {img.shape}")
    print(f"Caption shape: {caption.shape}")
    print(f"Label shape: {label.shape}")
    print(caption[0])
    print(label[0])

## モデル
画像キャプショニング モデルを設計しましょう。<br>
モデルは画像エンコーダと、それに続くキャプション デコーダで構成されます。

### 画像エンコーダ
画像エンコーダ モデルは非常にシンプルです。事前トレーニング済みモデルを介して特徴量を抽出し、全結合層に渡します。

1. この例では、InceptionResNetV2 の畳み込み層から特徴量を抽出し、(バッチサイズ, 8, 8, 1536) のベクトルを取得します。
1. ベクトルの形状を (バッチサイズ, 64, 1536) に変形（Reshape）します。
1. Dense レイヤーで `ATTENTION_DIM` の長さに圧縮し、(バッチサイズ, 64, ATTENTION_DIM) を返します。
1. 後ほど、Attention レイヤーが画像にアテンションを適用して次の単語を予測します。


In [ ]:
FEATURE_EXTRACTOR.trainable = False

image_input = Input(shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS))
image_features = FEATURE_EXTRACTOR(image_input)

x = Reshape((FEATURES_SHAPE[0] * FEATURES_SHAPE[1], FEATURES_SHAPE[2]))(
    image_features
)
encoder_output = Dense(ATTENTION_DIM, activation="relu")(x)

In [ ]:
encoder = keras.Model(inputs=image_input, outputs=encoder_output)
encoder.summary()

### キャプション デコーダ
キャプション デコーダには、入力画像のさまざまな部分に焦点を当てるアテンション機構が組み込まれています。


#### アテンション ヘッド

デコーダは、アテンションを使用して入力シーケンスの一部を選択的に注視します。
アテンションはサンプルごとに入力としてベクトルのシーケンスを受け取り、サンプルごとに「アテンション」ベクトルを返します。

仕組みを見てみましょう。

<img src="https://user-images.githubusercontent.com/6895245/173408554-d4b6387b-248b-421e-8911-550d0561d001.png" alt="attention equation 1" width="800">

<img src="https://user-images.githubusercontent.com/6895245/173408648-38c6b582-a68b-4697-982a-1d885b83dd0b.png" alt="attention equation 2" width="800">

各変数の意味:

* $s$: エンコーダのインデックス
* $t$: デコーダのインデックス
* $\alpha_{ts}$: アテンションの重み
* $h_s$: アテンションの対象となるエンコーダ出力のシーケンス（Transformer の用語ではアテンションの「key」および「value」）
* $h_t$: シーケンスにアテンションを向けるデコーダの状態（Transformer の用語ではアテンションの「query」）
* $c_t$: 結果として得られるコンテキスト ベクトル

数式の説明:

1. エンコーダの出力シーケンス全体に対する softmax として、アテンションの重み $\alpha_{ts}$ を計算します。
2. エンコーダ出力の加重和としてコンテキスト ベクトルを計算します。

最後に $score$ 関数です。その役割は、各 key-query ペアのスカラー ロジット スコアを計算することです。一般的なアプローチには以下の 2 つがあります。

<img src="https://user-images.githubusercontent.com/6895245/173408773-3781cacc-de00-49c6-9909-f6cd65a0501b.png" alt="attention equation 4" width="800">

このノートブックでは、事前定義された `layers.Attention` を使用して Luong スタイルのアテンションを実装します。

#### デコーダのステップ

デコーダの役割は、次の出力トークンの予測を生成することです。

1. デコーダは、現在の単語トークンをバッチとして受け取ります。
1. 単語トークンを `ATTENTION_DIM` 次元に埋め込みます（Embed）。
1. GRU レイヤーが単語の埋め込みを追跡し、GRU の出力と状態を返します。
1. Luong スタイルのアテンションが、GRU 出力をクエリとしてエンコーダの出力特徴量にアテンションを適用します。
1. アテンション出力と GRU 出力が加算され（スキップ接続）、レイヤー正規化（Layer Normalization）レイヤーで正規化されます。
1. GRU 出力に基づいて、次のトークンのロジット予測を生成します。

すべてのステップを Keras Functional API で定義できますが、推論フェーズでレイヤーと重みを再利用できるように、ここではトレーニング可能なパラメータを持つレイヤーをインスタンス化していることに注意してください。

**演習**: 上記の指示に従って、デコーダのステップを定義してください。

**参考資料**:
- [Embedding Layer](https://keras.io/api/layers/core_layers/embedding/)
- [GRU Layer](https://keras.io/api/layers/recurrent_layers/gru/)
- [Attention Layer](https://keras.io/api/layers/attention_layers/attention/)

In [ ]:
word_input = Input(shape=(MAX_CAPTION_LEN,), name="words")

# TODO: Embedding レイヤーを定義してください
embed_x = Embedding(...)(word_input)

# TODO: GRU レイヤーを定義してください。
decoder_gru = GRU(...)
gru_output, gru_state = decoder_gru(embed_x)

decoder_attention = Attention()
# TODO: Attention レイヤーへの入力を定義してください
context_vector = decoder_attention([...])

addition = Add()([gru_output, context_vector])

layer_norm = LayerNormalization(axis=-1)
layer_norm_out = layer_norm(addition)

decoder_output_dense = Dense(VOCAB_SIZE)
decoder_output = decoder_output_dense(layer_norm_out)

In [ ]:
decoder = keras.Model(
    inputs=[word_input, encoder_output], outputs=decoder_output
)
keras.utils.plot_model(decoder)

In [ ]:
decoder.summary()

### トレーニング用モデル

エンコーダとデコーダを定義しました。これらをトレーニング用の画像モデルに結合しましょう。<br>
このモデルには 2 つの入力（`image_input` と `word_input`）と 1 つの出力（`decoder_output`）があります。この定義は、データセット パイプラインの定義と一致している必要があります。

In [ ]:
image_caption_train_model = keras.Model(
    inputs=[image_input, word_input], outputs=decoder_output
)

### 損失関数
損失関数はシンプルなクロスエントロピーですが、計算時にパディング（`0`）を除外する必要があります。<br>
そのため、ここでは文の長さ（0 以外の部分）を抽出し、有効な文の部分についてのみ損失の平均を計算します。

In [ ]:
loss_object = keras.losses.SparseCategoricalCrossentropy(
    from_logits=True, reduction="none"
)


def loss_function(real, pred):
    loss_ = loss_object(real, pred)

    # 単語インデックスには 1、パディングには 0 を返す（例: [1,1,1,1,1,0,0,0,0,...,0]）
    mask = keras.ops.logical_not(keras.ops.equal(real, 0))
    mask = keras.ops.cast(mask, dtype="float32")
    loss_ *= mask

    return keras.ops.sum(loss_) / keras.ops.sum(mask)

In [ ]:
image_caption_train_model.compile(
    optimizer="adam",
    loss=loss_function,
)

## トレーニング ループ

標準の `model.fit` API を使用してモデルをトレーニングできるようになりました。<br>
NVIDIA T4 GPU を使用した場合、1 エポックのトレーニングに約 15〜20 分かかります。

In [ ]:
%%time
history = image_caption_train_model.fit(batched_ds, epochs=1)

## キャプションの生成

予測ステップはトレーニングとは異なります。キャプション生成中に GRU の状態を追跡し、予測された単語を次のタイムステップで入力としてデコーダに渡す必要があるためです。

そのために、トレーニング済みの重みを使用しながら、キャプション生成中に GRU の状態を保持・更新できるように、予測用の別のモデルを定義しましょう。

**演習**: 予測用のデコーダを完成させてください。

**ヒント**: アーキテクチャの大部分はトレーニング時と同じですが、この予測モデルには GRU 状態を受け渡すための追加入出力があります。

In [ ]:
# TODO: 上記のトレーニング用モデルを参考に、予測モデル全体を完成させてください。

gru_state_input = Input(shape=(ATTENTION_DIM,), name="gru_state_input")

# トレーニング済みの GRU を再利用し、状態を受け取れるように更新します。
gru_output, gru_state = decoder_gru(...)

# 他のレイヤーも同様に再利用します
context_vector = decoder_attention([...])
addition_output = Add()([...])
layer_norm_output = layer_norm(...)

decoder_output = decoder_output_dense(...)

# 状態の入力と出力を持つ予測モデルを定義
decoder_pred_model = keras.Model(
    inputs=[...],
    outputs=[...],
)



1. GRU の状態をゼロベクトルとして初期化します。
1. 入力画像を前処理し、エンコーダに渡して画像特徴量を抽出します。
1. キャプション生成を開始するために、`<start>` の単語トークンを設定します。
1. for ループ内で次の処理を行います。
    - 単語トークン（`dec_input`）、GRU 状態（`gru_state`）、画像特徴量（`features`）を予測デコーダに渡し、予測（`predictions`）と更新された GRU 状態を取得します。
    - ロジットから上位 K 個（Top-K）の単語を選択し、確率的に単語を選択することで、VOCAB_SIZE サイズのベクトルに対する softmax 計算を回避します。
    - モデルが `<end>` トークンを予測したら、予測を停止します。
    - 次のステップのために、入力単語トークンを予測された単語トークンに置き換えます。

In [ ]:
# トレーニング済みモデルを使用した確率的予測
def predict_caption(filename):
    gru_state = keras.ops.zeros((1, ATTENTION_DIM))

    img = tf.image.decode_jpeg(tf.io.read_file(filename), channels=IMG_CHANNELS)
    img = resize(img)
    img = rescale(img)

    features = encoder(keras.ops.expand_dims(img, axis=0))
    dec_input = keras.ops.expand_dims([word_to_index("<start>")], 1)
    result = []
    for i in range(MAX_CAPTION_LEN):
        predictions, gru_state = decoder_pred_model(
            [dec_input, gru_state, features]
        )

        # 予測によって与えられた対数分布からサンプリング
        top_probs, top_idxs = keras.ops.top_k(
            predictions[0][0], k=10, sorted=False
        )
        chosen_id = keras.random.categorical([top_probs], 1)[0].numpy()
        predicted_id = top_idxs.numpy()[chosen_id][0]

        result.append(tokenizer.get_vocabulary()[predicted_id])

        if predicted_id == word_to_index("<end>"):
            return img, result

        dec_input = keras.ops.expand_dims([predicted_id], 1)

    return img, result

キャプションを生成してみましょう。

In [ ]:
filename = "../sample_images/baseball.jpeg"  # surf.jpeg も試すことができます

for i in range(5):
    image, caption = predict_caption(filename)
    print(" ".join(caption[:-1]) + ".")

img = tf.image.decode_jpeg(tf.io.read_file(filename), channels=IMG_CHANNELS)
plt.imshow(img)
plt.axis("off");

いかがでしょうか。<br>
文法的に完全ではないかもしれませんが、人物、野球、ボールといった画像の重要な側面をモデルが捉えていることがわかります。

### オプションのタスク:
ご自身の画像をアップロードして、キャプションを生成してみてください。<br>
また、より長くトレーニングして、より優れたキャプションが得られるか試すこともできます。

## まとめ
画像エンコーダとテキスト デコーダを作成することで、画像キャプショニング モデルを構築する方法を学びました。

Copyright 2022 Google Inc. Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0 Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.